# Eclipse: Path of Totality

On 8 April 2024 the Moon's shadow crossed North America from Mexico's Pacific coast to Newfoundland. This tutorial computes where that eclipse was total: the **central line** and the **path of totality** between its **northern and southern limits**, and checks them against NASA's published predictions.

The Moon casts two shadow cones. The **penumbra**, which widens away from the Moon, covers the region where the Sun is partly hidden. The **umbra** narrows away from the Moon to a vertex; where the Earth's surface cuts it before the vertex, the Sun is completely hidden. Its footprint is an ellipse, typically 100 to 200 km across, that sweeps across the Earth as the Moon moves and the Earth turns. The swept area is the path of totality. The track of the shadow axis is the central line, and the two edges traced by the sides of the footprint are the northern and southern limits.

We use the classical method of **Besselian elements**, the one NASA and the national almanacs use ([Urban & Seidelmann 2013](../../guide/references/#urban2013); [Meeus 1989](../../guide/references/#meeus1989)). The idea is to describe the shadow in a plane where it is simple, fit a few smooth functions of time to that description, and only then map it onto the rotating, flattened Earth:

1. compute the Besselian elements from the JPL ephemerides in `satkit`, and compare them with NASA's;
2. map them to the ground: central line, limits, shadow outlines and duration of totality;
3. compare the result with NASA's path table;
4. reuse the same code for the next coast-to-coast eclipse, on 12 August 2045.

## Besselian elements

The **fundamental plane** passes through the Earth's centre, perpendicular to the shadow axis (the line through the centres of the Sun and the Moon). Its axes are

- $\hat{\mathbf z}$ along the shadow axis, pointing towards the Sun and Moon;
- $\hat{\mathbf x}$ in the Earth's equatorial plane, pointing east;
- $\hat{\mathbf y} = \hat{\mathbf z} \times \hat{\mathbf x}$, pointing north.

Lengths are in Earth equatorial radii ($a_e$ = 6378.137 km, WGS84). The shadow cones are circles on this plane, and the Earth turns underneath. Six numbers, all slowly varying functions of time, describe the geometry:

| Element | Meaning |
|---|---|
| $x$, $y$ | where the shadow axis crosses the fundamental plane |
| $d$ | declination of the shadow axis |
| $\mu$ | Greenwich hour angle of the shadow axis |
| $l_1$, $l_2$ | radii of the penumbral and umbral cones on the plane |
| $f_1$, $f_2$ | half-angles of the penumbral and umbral cones |

$l_2$ is negative when the umbral vertex lies beyond the plane, which is the case for a total eclipse; a positive $l_2$ means only the antumbra reaches the Earth (an annular eclipse). The sketch below shows the cones, not to scale.

In [ ]:
import datetime
import numpy as np
import satkit as sk
import pandas as pd

import ssl, certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']
import warnings
warnings.filterwarnings("ignore", "Downloading")
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
# Schematic of the shadow cones and the fundamental plane (not to scale)
sun_x, sun_r = 0.0, 2.0
moon_x, moon_r = 6.0, 0.8
earth_x, earth_r = 9.4, 1.1
D = moon_x - sun_x

# Umbra: tangent to the same side of Sun and Moon, converging to a vertex beyond the Moon
tan_f2 = (sun_r - moon_r) / np.sqrt(D**2 - (sun_r - moon_r) ** 2)
vertex2 = moon_x + moon_r / (sun_r - moon_r) * D
# Penumbra: tangents crossing between the Sun and the Moon
tan_f1 = (sun_r + moon_r) / np.sqrt(D**2 - (sun_r + moon_r) ** 2)
vertex1 = sun_x + sun_r / (sun_r + moon_r) * D
l1 = (earth_x - vertex1) * tan_f1
l2 = (vertex2 - earth_x) * tan_f2

fig, ax = plt.subplots(figsize=(9, 3.8))
for x, r, colour in [(sun_x, sun_r, "#EE7733"), (moon_x, moon_r, "#888888"), (earth_x, earth_r, "#0077BB")]:
    ax.add_patch(plt.Circle((x, 0), r, color=colour, alpha=0.3, lw=0))
ax.text(sun_x, -sun_r - 0.2, "Sun", ha="center", va="top", fontsize=11)
ax.text(moon_x, -moon_r - 0.3, "Moon", ha="center", va="top", fontsize=11)
ax.text(earth_x + 0.75, -earth_r + 0.05, "Earth", ha="left", va="top", fontsize=11)

xs = np.array([moon_x - 0.4, 11.2])
for s in (1, -1):
    ax.plot(xs, s * (vertex2 - xs) * tan_f2, color="k", lw=1.1)
    ax.plot(xs, s * (xs - vertex1) * tan_f1, color="k", lw=0.8, ls="--")
ax.plot([sun_x, 11.4], [0, 0], color="0.5", lw=0.6, ls=":")

# The fundamental plane passes through the Earth's centre, perpendicular to the shadow axis
ax.plot([earth_x, earth_x], [-3.0, 3.0], color="#009988", lw=1.5)
ax.text(earth_x - 0.1, 3.0, "fundamental plane", color="#009988", fontsize=10, va="top", ha="right")
ax.annotate("", xy=(earth_x + 0.15, l1), xytext=(earth_x + 0.15, 0), arrowprops=dict(arrowstyle="<->", lw=0.8))
ax.text(earth_x + 0.25, 0.55 * l1, "$l_1$", fontsize=12, va="center")
ax.annotate("$l_2$: umbral radius on the plane", xy=(earth_x, -l2), xytext=(earth_x + 0.4, -2.2),
            fontsize=9, arrowprops=dict(arrowstyle="->", lw=0.6))
ax.annotate("umbral vertex", xy=(vertex2, 0), xytext=(vertex2 + 0.2, 0.7), fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.6))
ax.annotate("", xy=(earth_x - 1.6, 0), xytext=(earth_x, 0), arrowprops=dict(arrowstyle="->", lw=1.0))
ax.text(earth_x - 1.6, 0.12, r"$\hat{z}$", fontsize=12, ha="center", va="bottom")
ax.annotate("umbra ($f_2$)", xy=(moon_x + 1.6, -0.25), xytext=(moon_x + 0.2, -2.5), fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.6))
ax.annotate("penumbra ($f_1$)", xy=(moon_x + 1.8, 1.3), xytext=(moon_x - 1.6, 2.4), fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.6))
ax.set_xlim(-2.2, 12.2)
ax.set_ylim(-3.1, 3.1)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## Computing the elements from the JPL ephemerides

`satkit.jplephem.geocentric_pos` gives the geocentric positions of the Sun and Moon. Two small corrections make them the positions the shadow geometry needs:

- **Sun: light time and aberration.** Sunlight arriving at the Earth left the Sun about 499 s earlier. Evaluating the geocentric Sun at $t - \tau_\odot$, with $\tau_\odot = |\mathbf r_\odot|/c$, applies that light time, and because the Earth's own position is also taken at $t-\tau_\odot$ it also applies the annual aberration (the Earth moves $\mathbf v_\oplus \tau_\odot$, about 15,000 km, in that time). This is the apparent Sun, as the [Eclipse](../Eclipse/) tutorial uses.
- **Moon: light time.** The light that grazes the Moon's limb reaches the Earth about 1.3 s later, so the Moon is taken at $t - \tau_{\rm M}$. In that time the shadow moves about 1 km.

Both vectors are rotated to the Earth-fixed ITRF with `satkit.frametransform.qgcrf2itrf`, which uses UT1 and polar motion from the IERS Earth orientation parameters. In that frame the shadow axis $\mathbf G = \mathbf r_\odot - \mathbf r_{\rm M}$ gives $d$ and $\mu$ directly, with $\mu$ measured from the Greenwich meridian:

$$
\hat{\mathbf z} = \frac{\mathbf G}{|\mathbf G|} = (\cos d\cos\mu,\ -\cos d\sin\mu,\ \sin d), \qquad
\hat{\mathbf x} = (\sin\mu,\ \cos\mu,\ 0), \qquad
\hat{\mathbf y} = (-\sin d\cos\mu,\ \sin d\sin\mu,\ \cos d).
$$

The Moon's coordinates on the fundamental plane are $x = \hat{\mathbf x}\cdot\mathbf r_{\rm M}$, $y = \hat{\mathbf y}\cdot\mathbf r_{\rm M}$ and $z = \hat{\mathbf z}\cdot\mathbf r_{\rm M}$. With the Sun's radius $R_\odot$ and the Moon's radius $k$ (both in $a_e$), the cones are

$$
\sin f_1 = \frac{R_\odot + k_1}{|\mathbf G|}, \qquad \sin f_2 = \frac{R_\odot - k_2}{|\mathbf G|}, \qquad
l_1 = \left(z + \frac{k_1}{\sin f_1}\right)\tan f_1, \qquad l_2 = \left(z - \frac{k_2}{\sin f_2}\right)\tan f_2 .
$$

**Radii.** For the Moon we use NASA's values ([Espenak](../../guide/references/#espenak)): $k_1 = 0.272488$ for the penumbra and $k_2 = 0.272281$ for the umbra. The IAU mean radius is $k = 0.2725076$ (1737.4 km). The smaller umbral value approximates the radius to the bottoms of the valleys on the lunar limb: sunlight shining through them (Baily's beads) ends totality early, so the mean radius would make the path about 3 km too wide. For the Sun we use NASA's semi-diameter of 959.63″ at 1 AU (696,000 km). `satkit.consts.sun_radius` is the IAU 2015 nominal photospheric radius, 695,700 km, which would widen the path by about 1.5 km.

Like the almanacs, we evaluate the elements every 10 minutes over six hours and fit a cubic polynomial in $t - t_0$ (hours) to each. The elements are smooth, so a cubic fits them to better than a metre.

In [ ]:
AE = sk.consts.wgs84_a                  # unit of length: Earth equatorial radius, m
FLAT = sk.consts.wgs84_f
K1, K2 = 0.272488, 0.272281             # lunar radius for the penumbra and umbra, Earth radii
ARCSEC = np.pi / 180 / 3600
R_SUN = sk.consts.au * np.tan(959.63 * ARCSEC) / AE  # solar radius, Earth radii


def sun_moon_itrf(times):
    # Apparent Sun and light-time-corrected Moon in the ITRF, in Earth radii
    c = sk.consts.c
    sun0 = sk.jplephem.geocentric_pos(sk.solarsystem.Sun, times)
    moon0 = sk.jplephem.geocentric_pos(sk.solarsystem.Moon, times)
    t_sun = [t - sk.duration.from_seconds(tau) for t, tau in zip(times, np.linalg.norm(sun0, axis=1) / c)]
    t_moon = [t - sk.duration.from_seconds(tau) for t, tau in zip(times, np.linalg.norm(moon0, axis=1) / c)]
    sun = sk.jplephem.geocentric_pos(sk.solarsystem.Sun, t_sun)
    moon = sk.jplephem.geocentric_pos(sk.solarsystem.Moon, t_moon)
    q = sk.frametransform.qgcrf2itrf(times)
    sun = np.array([qi * v for qi, v in zip(q, sun)]) / AE
    moon = np.array([qi * v for qi, v in zip(q, moon)]) / AE
    return sun, moon


def raw_elements(times):
    # Besselian elements at each time (d and mu in degrees, lengths in Earth radii)
    sun, moon = sun_moon_itrf(times)
    G = sun - moon
    Gn = np.linalg.norm(G, axis=1)
    zhat = G / Gn[:, None]
    d = np.arcsin(zhat[:, 2])
    mu = np.unwrap(np.arctan2(-zhat[:, 1], zhat[:, 0]))
    xhat = np.stack([np.sin(mu), np.cos(mu), np.zeros_like(mu)], axis=1)
    yhat = np.stack([-np.sin(d) * np.cos(mu), np.sin(d) * np.sin(mu), np.cos(d)], axis=1)
    x, y, z = (np.sum(v * moon, axis=1) for v in (xhat, yhat, zhat))
    sin_f1 = (R_SUN + K1) / Gn
    sin_f2 = (R_SUN - K2) / Gn
    tan_f1 = sin_f1 / np.sqrt(1 - sin_f1**2)
    tan_f2 = sin_f2 / np.sqrt(1 - sin_f2**2)
    return {
        "x": x, "y": y, "d": np.degrees(d), "mu": np.degrees(mu),
        "l1": (z + K1 / sin_f1) * tan_f1, "l2": (z - K2 / sin_f2) * tan_f2,
        "tan_f1": tan_f1, "tan_f2": tan_f2,
    }


class Besselian:
    # Cubic polynomials in t - t0 (hours, TT) for x, y, d, l1, l2 and mu
    names = ["x", "y", "d", "l1", "l2", "mu"]
    P = np.polynomial.polynomial

    def __init__(self, t0, coef, tan_f1, tan_f2):
        self.t0, self.coef, self.tan_f1, self.tan_f2 = t0, coef, tan_f1, tan_f2

    @classmethod
    def fit(cls, t0, span_hours=3.0, step_hours=1 / 6):
        th = np.arange(-span_hours, span_hours + 1e-9, step_hours)
        e = raw_elements([t0 + sk.duration.from_hours(h) for h in th])
        coef = {k: cls.P.polyfit(th, e[k], 3) for k in cls.names}
        coef["mu"][0] %= 360
        return cls(t0, coef, e["tan_f1"].mean(), e["tan_f2"].mean())

    def __call__(self, th):
        # Elements and their rates (per hour) at hours th from t0
        out = {k: self.P.polyval(th, c) for k, c in self.coef.items()}
        out.update({k + "_dot": self.P.polyval(th, self.P.polyder(c)) for k, c in self.coef.items()})
        return out

    def hours(self, t):
        # Hours from t0 of a satkit time
        return (t - self.t0).hours

    def time(self, th):
        # satkit time at hours th from t0
        return self.t0 + sk.duration.from_hours(float(th))


# t0 = 2024-04-08 18:00 TT, NASA's reference epoch for this eclipse
t0 = sk.time.from_jd(2460409.25, sk.timescale.TT)
bes = Besselian.fit(t0)
print(f"t0 = {t0} UTC (18:00:00 TT)")

## Comparison with NASA's elements

NASA publishes polynomial Besselian elements for each eclipse (Fred Espenak, [eclipse.gsfc.nasa.gov, SE2024Apr08T Besselian elements](https://eclipse.gsfc.nasa.gov/SEbeselm/SEbeselm2001/SE2024Apr08Tbeselm.html)); the coefficients are copied below, with $\tan f_1 = 0.0046683$ and $\tan f_2 = 0.0046450$. They are computed from the VSOP87 and ELP2000-85 analytical theories rather than the JPL numerical ephemerides.

**Time scales.** NASA's polynomials use Terrestrial Dynamical Time, $t_0$ = 18:00 TT, which is also our $t_0$. The Earth's rotation, however, follows UT1, and $\Delta T = \mathrm{TT} - \mathrm{UT1}$ was not known when the predictions were made; NASA assumed $\Delta T$ = 70.6 s. Following the almanac convention, NASA's $\mu$ is the *ephemeris* hour angle, computed as if UT1 were TT, and the Greenwich hour angle is

$$ \mu_{\rm Greenwich} = \mu - 1.002738 \times 15''\,{\rm s}^{-1} \times \Delta T = \mu - 0.00417807^\circ \times \Delta T[{\rm s}] .$$

Our $\mu$ comes directly from the ITRF rotation, so it already uses the measured UT1. In April 2024, $\Delta T$ was 69.2 s; NASA's value was 1.4 s too large, which moves its path about 0.5 km east. Below we convert NASA's $\mu$ with the measured $\Delta T$, so that the comparison shows the ephemeris differences only, and then list the effect of the $\Delta T$ difference separately. Lengths are converted to kilometres ($\times\,a_e$).

In [ ]:
# NASA / Espenak polynomial Besselian elements, 2024 Apr 08, t0 = 18:00 TDT, Delta T = 70.6 s
# "Eclipse Predictions by Fred Espenak, NASA's GSFC"
# https://eclipse.gsfc.nasa.gov/SEbeselm/SEbeselm2001/SE2024Apr08Tbeselm.html
NASA_COEF = {
    "x": [-0.318157, 0.5117105, 0.0000326, -0.0000085],
    "y": [0.219747, 0.2709586, -0.0000594, -0.0000047],
    "d": [7.58620, 0.014844, -0.000002, 0.0],
    "l1": [0.535813, 0.0000618, -0.0000128, 0.0],
    "l2": [-0.010274, 0.0000615, -0.0000127, 0.0],
    "mu": [89.59122, 15.004084, 0.0, 0.0],
}
NASA_TAN_F1, NASA_TAN_F2 = 0.0046683, 0.0046450
NASA_DELTA_T = 70.6


def delta_t(t):
    # TT - UT1 in seconds, from satkit's leap seconds and the IERS UT1 - UTC
    tt_minus_utc = (t.to_mjd(sk.timescale.TT) - t.to_mjd(sk.timescale.UTC)) * 86400
    return tt_minus_utc - sk.frametransform.earth_orientation_params(t)[0]


def nasa_besselian(delta_t_seconds):
    # NASA's elements with mu converted to the Greenwich hour angle for a given Delta T
    coef = {k: np.array(v, dtype=float) for k, v in NASA_COEF.items()}
    coef["mu"][0] -= 0.00417807 * delta_t_seconds
    return Besselian(t0, coef, NASA_TAN_F1, NASA_TAN_F2)


dT = delta_t(t0)
print(f"Delta T from satkit: {dT:.2f} s (NASA assumed {NASA_DELTA_T} s)")
nasa_measured_dt = nasa_besselian(dT)
nasa_own_dt = nasa_besselian(NASA_DELTA_T)

th = np.arange(-3.0, 3.01, 1.0)
ours, theirs, theirs_own = bes(th), nasa_measured_dt(th), nasa_own_dt(th)
km = AE / 1e3
df = pd.DataFrame({
    "TT": [f"{18 + h:02.0f}:00" for h in th],
    "x (km)": (ours["x"] - theirs["x"]) * km,
    "y (km)": (ours["y"] - theirs["y"]) * km,
    "d (arcsec)": (ours["d"] - theirs["d"]) * 3600,
    "mu (arcsec)": (ours["mu"] - theirs["mu"]) * 3600,
    "l1 (km)": (ours["l1"] - theirs["l1"]) * km,
    "l2 (km)": (ours["l2"] - theirs["l2"]) * km,
    "mu, NASA's Delta T (arcsec)": (ours["mu"] - theirs_own["mu"]) * 3600,
})
print(f"tan f1: satkit {bes.tan_f1:.7f}, NASA {NASA_TAN_F1:.7f}")
print(f"tan f2: satkit {bes.tan_f2:.7f}, NASA {NASA_TAN_F2:.7f}")
print("satkit minus NASA:")
df.style.format(precision=3).hide(axis="index")

The elements agree closely:

- $x$ and $y$, the shadow axis position, differ by about 0.6 km, and the difference barely changes over the six hours. That is 0.3″ as seen from the Earth, the expected level of disagreement between the ELP2000-85 lunar theory and the JPL DE440 ephemeris for 2024.
- $d$ differs by about 0.4″. Our frame is the ITRF, whose pole differs from the celestial pole NASA uses by the polar motion, about 0.36″ on that day; this is a difference of reference, worth about 10 m on the ground.
- $\mu$ agrees to a few hundredths of an arcsecond once both use the measured $\Delta T$. With NASA's $\Delta T$ the difference is 21″, the 1.4 s of Earth rotation noted above.
- $l_1$, $l_2$, $\tan f_1$ and $\tan f_2$ agree to 30 m and to the last digit NASA gives: the shadow cones are the same.

## From the elements to the ground

**Observer coordinates.** A point $\mathbf p$ on the Earth (ITRF, in $a_e$) has fundamental-plane coordinates $\xi = \hat{\mathbf x}\cdot\mathbf p$, $\eta = \hat{\mathbf y}\cdot\mathbf p$, $\zeta = \hat{\mathbf z}\cdot\mathbf p$; $\zeta$ is its height above the plane, towards the Sun. It is inside the umbra when its distance from the shadow axis is less than the umbral radius at that height:

$$ (x-\xi)^2 + (y-\eta)^2 < L_2^2, \qquad L_2 = l_2 - \zeta \tan f_2 .$$

**Back to the ellipsoid.** Going the other way, given a point $(\xi, \eta)$ on the plane, we need the $\zeta$ at which the line $\xi\hat{\mathbf x} + \eta\hat{\mathbf y} + \zeta\hat{\mathbf z}$ meets the WGS84 ellipsoid on the side facing the Sun. Stretching the ITRF $z$ axis by $1/(1-f)$ turns the ellipsoid into the unit sphere, so $\zeta$ is the larger root of a quadratic; if there is no real root, the point is off the Earth's limb. (The Explanatory Supplement does the same with its auxiliary quantities $\rho_1$, $\rho_2$, $d_1$ and $d_2$.) With $\zeta$ known, $\mathbf p$ is converted to latitude and longitude with `satkit.itrfcoord`.

**Central line.** The shadow axis meets the ground at $(\xi, \eta) = (x, y)$.

**Limits.** At each instant the umbra is a circle of radius $|L_2|$ around $(x, y)$. The limit lines are the envelope of these circles as they sweep over the ground, and each circle touches the envelope where its radius is perpendicular to the shadow's motion *relative to the observer*. The observer moves on the plane because the Earth turns:

$$ \dot\xi = \dot\mu\,(\zeta\cos d - \eta\sin d), \qquad \dot\eta = \dot\mu\,\xi\sin d - \dot d\,\zeta, $$

so the relative velocity is $(a, b) = (\dot x - \dot\xi,\ \dot y - \dot\eta)$, with $n = \sqrt{a^2+b^2}$ ($\dot\mu$ and $\dot d$ in radians per hour). The limit points are

$$ \xi = x \mp |L_2|\,\frac{b}{n}, \qquad \eta = y \pm |L_2|\,\frac{a}{n} $$

(upper signs north). $L_2$ and the observer's velocity depend on the $\zeta$ of the point being sought, so we iterate: start from the central line, compute the limit point, find its $\zeta$ on the ellipsoid, and repeat. Eight iterations converge to a few centimetres, even near sunrise and sunset, where convergence is slowest. The limit lines are the outline of the union of all the footprints; the tangency condition finds that outline exactly at each instant, instead of to the spacing of a set of drawn footprints, and gives each limit point a time, as in NASA's tables.

**Duration of totality.** For a fixed place, Newton's method on $t$ with the step $\Delta t = -(u a + v b)/n^2$, where $(u, v) = (x-\xi,\ y-\eta)$, finds the moment the observer is closest to the shadow axis. The closest distance is $m = |u b - v a|/n$, and totality lasts

$$ \tau = \frac{2\sqrt{L_2^2 - m^2}}{n} $$

when $m < |L_2|$ and the Sun is above the horizon ($\zeta > 0$). Evaluated on a grid of places, this gives contours of equal duration.

These are the formulas of the Besselian-elements chapter of the Explanatory Supplement ([Urban & Seidelmann 2013](../../guide/references/#urban2013)) and of [Meeus (1989)](../../guide/references/#meeus1989), vectorised with numpy.

In [ ]:
STRETCH = np.array([1.0, 1.0, 1.0 / (1.0 - FLAT)])  # maps the WGS84 ellipsoid to the unit sphere


def plane_axes(e):
    # ITRF unit vectors of the fundamental-plane axes, shape (..., 3)
    d, mu = np.radians(e["d"]), np.radians(e["mu"])
    xhat = np.stack([np.sin(mu), np.cos(mu), np.zeros_like(mu)], axis=-1)
    yhat = np.stack([-np.sin(d) * np.cos(mu), np.sin(d) * np.sin(mu), np.cos(d)], axis=-1)
    zhat = np.stack([np.cos(d) * np.cos(mu), -np.cos(d) * np.sin(mu), np.sin(d)], axis=-1)
    return xhat, yhat, zhat


def surface_zeta(xi, eta, axes):
    # zeta at which (xi, eta) meets the ellipsoid on the sunward side; NaN off the limb
    xhat, yhat, zhat = axes
    a = (xi[..., None] * xhat + eta[..., None] * yhat) * STRETCH
    b = zhat * STRETCH
    qa, qb, qc = np.sum(b * b, -1), np.sum(a * b, -1), np.sum(a * a, -1) - 1
    with np.errstate(invalid="ignore"):
        return (-qb + np.sqrt(qb**2 - qa * qc)) / qa


def observer_rates(e, xi, eta, zeta):
    # Velocity of a point fixed on the Earth, in fundamental-plane coordinates per hour
    d = np.radians(e["d"])
    mu_dot, d_dot = np.radians(e["mu_dot"]), np.radians(e["d_dot"])
    return mu_dot * (zeta * np.cos(d) - eta * np.sin(d)), mu_dot * xi * np.sin(d) - d_dot * zeta


def to_itrf(xi, eta, zeta, axes):
    xhat, yhat, zhat = axes
    return (xi[..., None] * xhat + eta[..., None] * yhat + zeta[..., None] * zhat) * AE


def central_line(bes, th):
    e = bes(th)
    axes = plane_axes(e)
    return to_itrf(e["x"], e["y"], surface_zeta(e["x"], e["y"], axes), axes)


def limit_lines(bes, th, niter=8):
    # Northern and southern limits of the umbra at hours th (ITRF, m; NaN where off the Earth)
    e = bes(th)
    axes = plane_axes(e)
    out = {}
    for side, sign in (("north", 1), ("south", -1)):
        xi, eta = e["x"], e["y"]
        zeta = np.nan_to_num(surface_zeta(xi, eta, axes))
        for _ in range(niter):
            xi_dot, eta_dot = observer_rates(e, xi, eta, zeta)
            a, b = e["x_dot"] - xi_dot, e["y_dot"] - eta_dot
            n = np.hypot(a, b)
            L2 = np.abs(e["l2"] - zeta * bes.tan_f2)
            xi, eta = e["x"] - sign * L2 * b / n, e["y"] + sign * L2 * a / n
            zeta = surface_zeta(xi, eta, axes)
        out[side] = to_itrf(xi, eta, zeta, axes)
    return out


def umbra_outline(bes, th, npts=181, niter=6):
    # Outline of the umbra on the ground at one instant (ITRF, m)
    e = bes(np.array([th]))
    axes = plane_axes(e)
    q = np.linspace(0, 2 * np.pi, npts)
    zeta = np.zeros_like(q)
    for _ in range(niter):
        L2 = np.abs(e["l2"] - np.nan_to_num(zeta) * bes.tan_f2)
        xi, eta = e["x"] + L2 * np.cos(q), e["y"] + L2 * np.sin(q)
        zeta = surface_zeta(xi, eta, axes)
    return to_itrf(xi, eta, zeta, axes)


def totality_duration(bes, p_itrf, niter=6):
    # Duration of totality (s) and hours from t0 of maximum eclipse at ITRF points (m)
    p = np.asarray(p_itrf, dtype=float) / AE
    th = np.zeros(p.shape[:-1])
    with np.errstate(invalid="ignore", divide="ignore"):
        return _duration_newton(bes, p, th, niter)


def _duration_newton(bes, p, th, niter):
    for _ in range(niter):
        e = bes(th)
        xi, eta, zeta = (np.sum(p * v, -1) for v in plane_axes(e))
        xi_dot, eta_dot = observer_rates(e, xi, eta, zeta)
        u, v = e["x"] - xi, e["y"] - eta
        a, b = e["x_dot"] - xi_dot, e["y_dot"] - eta_dot
        n2 = a * a + b * b
        th = th - (u * a + v * b) / n2
    m = np.abs(u * b - v * a) / np.sqrt(n2)
    L2 = np.abs(e["l2"] - zeta * bes.tan_f2)
    tau = np.where((m < L2) & (zeta > 0), 2 * np.sqrt(np.abs(L2**2 - m**2) / n2) * 3600, np.nan)
    return tau, th


def geodetic_grid(lon_deg, lat_deg):
    # ITRF positions (m) of a lat/lon grid on the ellipsoid
    lon, lat = np.meshgrid(np.radians(lon_deg), np.radians(lat_deg))
    e2 = FLAT * (2 - FLAT)
    N = AE / np.sqrt(1 - e2 * np.sin(lat) ** 2)
    return np.stack([N * np.cos(lat) * np.cos(lon), N * np.cos(lat) * np.sin(lon),
                     N * (1 - e2) * np.sin(lat)], axis=-1)


def latlon(p):
    # Geodetic latitude and longitude (deg) of ITRF points; NaN rows stay NaN
    flat = p.reshape(-1, 3)
    out = np.full((len(flat), 2), np.nan)
    ok = np.all(np.isfinite(flat), axis=1)
    if ok.any():
        out[ok] = [(c.latitude_deg, c.longitude_deg) for c in map(sk.itrfcoord, flat[ok])]
    return out.reshape(*p.shape[:-1], 2)

## The 2024 path

Evaluate the elements every 2 seconds over the six-hour window; the shadow is on the Earth for about 3 h 17 min of it. Points off the Earth come back as NaN and are dropped, which handles the ends of the path, where only part of the umbra touches the Earth at sunrise and sunset. The code uses $|L_2|$, so for an annular eclipse ($L_2 > 0$, the vertex above the ground) the same functions give the path of annularity; the sign of $L_2$ tells which one it is, and changes along the path of a hybrid eclipse. We also evaluate the duration of totality on a 0.1° grid for the contours.

In [ ]:
def greatest_eclipse(bes):
    # Minimum distance of the shadow axis from the Earth's centre (gamma) and its time
    th = np.linspace(-3, 3, 21601)
    e = bes(th)
    gamma = np.hypot(e["x"], e["y"])
    i = np.argmin(gamma)
    return gamma[i], bes.time(th[i])


def compute_path(bes, step_seconds=2.0, grid=None):
    th = np.arange(-3.0, 3.0, step_seconds / 3600)
    path = {"bes": bes, "th": th, "central": central_line(bes, th)}
    path.update(limit_lines(bes, th))
    path["ll"] = {k: latlon(path[k]) for k in ("central", "north", "south")}
    if grid is not None:
        lon, lat = grid
        tau, _ = totality_duration(bes, geodetic_grid(lon, lat))
        path["grid"] = (lon, lat, tau)
    return path


path2024 = compute_path(bes, grid=(np.arange(-125, -50, 0.1), np.arange(14, 54, 0.1)))
ok = np.isfinite(path2024["ll"]["central"][:, 0])
t_first, t_last = (bes.time(path2024["th"][i]) for i in np.flatnonzero(ok)[[0, -1]])
print(f"central line on the Earth from {t_first.strftime('%H:%M:%S')} to {t_last.strftime('%H:%M:%S')} UTC")
gamma, t_ge = greatest_eclipse(bes)
print(f"greatest eclipse at {t_ge.strftime('%H:%M:%S.%f')[:10]} UTC, gamma = {gamma:.4f} "
      "(NASA: 18:17:18.3 UT with its Delta T, gamma = 0.3431)")

## Map

The map below shows the path of totality over North America: the band between the limits, the central line with the time (UTC) every 10 minutes, the outline of the umbra every 20 minutes, and contours of equal duration of totality, one colour per minute (the path is only about 40 pixels wide at this scale, too narrow for labels on the contours). The cities are those of the [Eclipse](../Eclipse/) tutorial, labelled with the duration of totality where they saw it. The inset shows the whole track.

In [ ]:
PC = ccrs.PlateCarree()
LAMBERT = ccrs.LambertConformal(central_longitude=-96, central_latitude=39, standard_parallels=(33, 45))
LAND, OCEAN = "#f2efe6", "#dfe9f1"
BAND, CENTRE, OUTLINE = "#4a5878", "#CC3311", "#222222"
CONTOUR_COLORS = ["#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c", "#08306b"]  # 1..6 min
LABEL_BOX = dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.75)


def fmt_duration(s):
    return f"{int(s // 60)}m{int(round(s % 60)):02d}s"


def base_map(ax, extent):
    # Lambert conformal map of North America with Natural Earth 1:50m features
    ax.set_extent(extent, crs=PC)
    ax.set_facecolor(OCEAN)
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor=LAND, edgecolor="none").set_rasterized(True)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor=OCEAN, edgecolor="#9aa8b3", linewidth=0.3)
    ax.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="#a39d8f", linewidth=0.35, facecolor="none")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), edgecolor="#555555", linewidth=0.7)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="#6b7a86", linewidth=0.5)
    gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False, rotate_labels=False,
                      linewidth=0.3, color="#888888", alpha=0.5, linestyle="--")
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 9}


def utc_marks(bes, th, minutes):
    # (hours from t0, datetime) at each whole multiple of `minutes` UTC within th
    first, last = bes.time(th[0]).to_datetime(), bes.time(th[-1]).to_datetime()
    mark = first.replace(second=0, microsecond=0)
    mark += datetime.timedelta(minutes=(-mark.minute) % minutes)
    marks = []
    while mark <= last:
        marks.append((bes.hours(sk.time.from_datetime(mark)), mark))
        mark += datetime.timedelta(minutes=minutes)
    return marks


def draw_path(ax, path, cities=(), tick_minutes=10, outline_minutes=20, tick_side="south"):
    bes, th = path["bes"], path["th"]
    ll = {k: v[::5] for k, v in path["ll"].items()}  # every 10 s is plenty for drawing
    both = np.isfinite(ll["north"][:, 0]) & np.isfinite(ll["south"][:, 0])
    band = np.concatenate([ll["north"][both], ll["south"][both][::-1]])
    ax.fill(band[:, 1], band[:, 0], transform=PC, color=BAND, alpha=0.25, lw=0, label="path of totality")
    for side in ("north", "south"):
        ax.plot(ll[side][:, 1], ll[side][:, 0], transform=PC, color=BAND, lw=0.8)
    ax.plot(ll["central"][:, 1], ll["central"][:, 0], transform=PC, color=CENTRE, lw=1.4,
            label="central line")

    # Contours of equal duration of totality, one colour per minute
    lon, lat, tau = path["grid"]
    levels = np.arange(1, int(np.nanmax(tau) // 60) + 1)
    ax.contour(lon, lat, tau / 60, levels=levels, transform=PC, linewidths=0.7,
               colors=CONTOUR_COLORS[: len(levels)])
    for lev in levels:
        ax.plot([], [], color=CONTOUR_COLORS[lev - 1], lw=1.2, label=f"{lev} min of totality")

    # Ticks across the central line every `tick_minutes`, labelled beyond one limit
    proj = ax.projection
    x0, x1, y0, y1 = ax.get_extent(crs=proj)
    for h, when in utc_marks(bes, th, tick_minutes):
        lim = limit_lines(bes, np.array([h]))
        pts = np.array([central_line(bes, np.array([h]))[0], lim["north"][0], lim["south"][0]])
        if not np.all(np.isfinite(pts)):
            continue
        c, n, s = proj.transform_points(PC, *latlon(pts)[:, ::-1].T)[:, :2]
        if not (x0 < c[0] < x1 and y0 < c[1] < y1):
            continue
        edge = s if tick_side == "south" else n
        ax.plot(*np.array([c + 0.35 * (n - c), c + 0.35 * (s - c)]).T, transform=proj, color=CENTRE, lw=1.0)
        ax.text(*(c + 1.55 * (edge - c)), when.strftime("%H:%M"), transform=proj, fontsize=8,
                ha="center", va="center", color=CENTRE, bbox=LABEL_BOX)
        if outline_minutes and when.minute % outline_minutes == 0:
            o = latlon(umbra_outline(bes, h))
            ax.plot(o[:, 1], o[:, 0], transform=PC, color=OUTLINE, lw=0.6, ls=(0, (3, 2)))
    if outline_minutes:
        ax.plot([], [], color=OUTLINE, lw=0.6, ls=(0, (3, 2)), label=f"umbra every {outline_minutes} min")

    # Cities: (name, lat, lon, (label dx, dy in degrees, horizontal alignment))
    for name, lat, lon, (dx, dy, ha) in cities:
        tau_city, _ = totality_duration(bes, sk.itrfcoord(latitude_deg=lat, longitude_deg=lon).vector)
        label = name if np.isnan(tau_city) else f"{name} {fmt_duration(tau_city)}"
        ax.plot(lon, lat, "o", transform=PC, ms=3.5, color="k", mec="white", mew=0.6, zorder=5)
        ax.text(lon + dx, lat + dy, label, transform=PC, fontsize=8, ha=ha, va="center",
                bbox=LABEL_BOX, zorder=5)


def inset_globe(fig, path, rect, centre):
    # Orthographic view of the whole track
    ax = fig.add_axes(rect, projection=ccrs.Orthographic(*centre))
    ax.set_global()
    ax.set_facecolor(OCEAN)
    ax.add_feature(cfeature.LAND, facecolor=LAND, edgecolor="none")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor="#6b7a86")
    ll = path["ll"]
    both = np.isfinite(ll["north"][:, 0]) & np.isfinite(ll["south"][:, 0])
    band = np.concatenate([ll["north"][both], ll["south"][both][::-1]])
    ax.fill(band[:, 1], band[:, 0], transform=PC, color=BAND, alpha=0.6, lw=0)
    ax.plot(ll["central"][:, 1], ll["central"][:, 0], transform=PC, color=CENTRE, lw=0.8)
    ax.spines["geo"].set_linewidth(0.5)
    return ax


# The cities of the Eclipse tutorial's statistics table, with label offsets
CITIES_2024 = [
    ("Mexico City", 19.4326, -99.1332, (0.5, 0, "left")),
    ("Austin", 30.2672, -97.7431, (0.5, -0.3, "left")),
    ("Dallas", 32.7767, -96.7970, (-0.5, 0.2, "right")),
    ("St. Louis", 38.6270, -90.1994, (-0.5, 0.2, "right")),
    ("Cleveland", 41.4993, -81.6944, (-0.5, 0.4, "right")),
    ("Burlington", 44.4759, -73.2121, (-0.5, -0.3, "right")),
    ("Montreal", 45.5017, -73.5673, (-0.5, 0.3, "right")),
    ("Quebec City", 46.8139, -71.2080, (0.4, 0.6, "left")),
    ("New York", 40.7128, -74.0060, (0.5, -0.2, "left")),
    ("Boston", 42.3601, -71.0589, (0.5, -0.2, "left")),
    ("Halifax", 44.6488, -63.5752, (0.5, -0.2, "left")),
]

fig, ax = plt.subplots(figsize=(10, 7.4), subplot_kw={"projection": LAMBERT})
base_map(ax, [-116, -62, 17, 50])
draw_path(ax, path2024, CITIES_2024)
ax.legend(loc="upper left", fontsize=8.5, framealpha=0.9)
ax.set_title("Total solar eclipse of 8 April 2024: path of totality")
inset_globe(fig, path2024, [0.67, 0.12, 0.21, 0.21], (-90, 25))
plt.show()

## Comparison with NASA's path table

NASA's path table ([Espenak, SE2024Apr08T path](https://eclipse.gsfc.nasa.gov/SEpath/SEpath2001/SE2024Apr08Tpath.html)) lists the northern limit, southern limit and central line every two minutes, with the path width and the duration of totality on the central line. The eight rows below, between Mexico and Maine, are copied from it. Each of NASA's points belongs to an instant in UT computed with $\Delta T$ = 70.6 s, so a timing difference moves a point *along* the path. What matters for an observer is the distance *across* it, so for each of NASA's points we give its distance from our corresponding line (positive when our line lies north of NASA's point). The width is measured across the path through our central point at the same UT, and the duration is ours at NASA's central-line point.

Two checks separate the error sources:

- **Mapping only**: our mapping from elements to the ground, fed NASA's own elements and $\Delta T$, against NASA's points. This tests the formulas; NASA's table is rounded to 0.1′ (up to 185 m).
- **satkit**: the whole calculation, with the JPL ephemerides and the measured $\Delta T$.

In [ ]:
# NASA / Espenak path table, 2024 Apr 08 ("Eclipse Predictions by Fred Espenak, NASA's GSFC")
# https://eclipse.gsfc.nasa.gov/SEpath/SEpath2001/SE2024Apr08Tpath.html
# UT, northern limit, southern limit, central line (deg, arcmin, N/S, deg, arcmin, E/W),
# path width (km), central duration (s)
NASA_PATH = [
    ("18:10", (23, 47.7, "N", 106, 50.7, "W"), (22, 36.2, "N", 105, 23.3, "W"), (23, 11.9, "N", 106, 6.8, "W"), 199, 267.2),
    ("18:30", (29, 32.2, "N", 101, 11.9, "W"), (28, 15.1, "N", 99, 49.8, "W"), (28, 53.6, "N", 100, 30.7, "W"), 195, 267.0),
    ("18:40", (32, 23.5, "N", 98, 1.3, "W"), (31, 3.1, "N", 96, 43.3, "W"), (31, 43.2, "N", 97, 22.2, "W"), 192, 263.7),
    ("18:50", (35, 14.1, "N", 94, 26.9, "W"), (33, 50.1, "N", 93, 14.6, "W"), (34, 32.0, "N", 93, 50.7, "W"), 189, 258.0),
    ("19:00", (38, 3.8, "N", 90, 18.7, "W"), (36, 36.0, "N", 89, 14.5, "W"), (37, 19.7, "N", 89, 46.6, "W"), 186, 250.0),
    ("19:10", (40, 51.6, "N", 85, 22.6, "W"), (39, 19.9, "N", 84, 29.8, "W"), (40, 5.6, "N", 84, 56.3, "W"), 183, 239.4),
    ("19:20", (43, 35.6, "N", 79, 16.9, "W"), (42, 0.4, "N", 78, 40.9, "W"), (42, 47.8, "N", 78, 59.2, "W"), 179, 225.9),
    ("19:30", (46, 11.6, "N", 71, 24.2, "W"), (44, 33.8, "N", 71, 13.8, "W"), (45, 22.4, "N", 71, 19.7, "W"), 173, 209.1),
]


def nasa_point(p):
    lat = (p[0] + p[1] / 60) * (1 if p[2] == "N" else -1)
    lon = (p[3] + p[4] / 60) * (1 if p[5] == "E" else -1)
    return sk.itrfcoord(latitude_deg=lat, longitude_deg=lon)


def offset_km(point, line):
    # Signed distance (km) from an itrfcoord to a polyline of ITRF points; + if the line is north
    L = line[np.all(np.isfinite(line), axis=1)]
    p = point.vector
    i = np.argmin(np.linalg.norm(L - p, axis=1))
    candidates = []
    for j in (max(i - 1, 0), min(i, len(L) - 2)):
        a, b = L[j], L[j + 1]
        s = np.clip(np.dot(p - a, b - a) / np.dot(b - a, b - a), 0, 1)
        candidates.append(a + s * (b - a))
    q = min(candidates, key=lambda c: np.linalg.norm(c - p))
    north = point.qenu2itrf * np.array([0.0, 1.0, 0.0])
    return np.sign(np.dot(q - p, north)) * np.linalg.norm(q - p) / 1e3


# NASA's elements with NASA's Delta T, mapped with our formulas, as lines
nasa_path = compute_path(nasa_own_dt)

rows = []
for ut, pn, ps, pc, width, dur in NASA_PATH:
    hh, mm = map(int, ut.split(":"))
    t_ut = sk.time(2024, 4, 8, hh, mm, 0)
    N, S, C = nasa_point(pn), nasa_point(ps), nasa_point(pc)
    # our width, across the path through our central point at this UT
    c = sk.itrfcoord(central_line(bes, np.array([bes.hours(t_ut)]))[0])
    our_width = abs(offset_km(c, path2024["north"])) + abs(offset_km(c, path2024["south"]))
    tau, _ = totality_duration(bes, C.vector[None, :])
    rows.append({
        "UT": ut,
        "mapping only: max |offset| (km)": max(abs(offset_km(P, nasa_path[k]))
                                               for P, k in ((N, "north"), (C, "central"), (S, "south"))),
        "north limit (km)": offset_km(N, path2024["north"]),
        "central line (km)": offset_km(C, path2024["central"]),
        "south limit (km)": offset_km(S, path2024["south"]),
        "width NASA (km)": width,
        "width satkit (km)": our_width,
        "duration NASA (s)": dur,
        "duration satkit (s)": tau[0],
    })
pd.DataFrame(rows).style.format(precision=2).format(precision=1, subset=["width satkit (km)", "duration NASA (s)", "duration satkit (s)"]).hide(axis="index")

With NASA's elements, our mapping reproduces NASA's points to the rounding of the table, so the formulas agree. The full satkit calculation puts all three lines about 0.5 to 1 km north (more precisely, north-west, across the path) of NASA's, with the same width to within a kilometre and the same central duration to a fraction of a second. The shift has two parts of similar size:

- **$\Delta T$.** The Earth turned 1.4 s further than NASA assumed, which moves the shadow track 21″ of longitude (about 0.5 km) west; across a path heading north-east, most of that appears as a shift to the north-west.
- **Ephemeris.** The shadow axis ($x$) sits 0.6 km further west on the fundamental plane with DE440 than with ELP2000-85, which also moves the path north-west.

Both are at the level of the other things neither calculation includes. The real lunar limb is not a circle: its mountains and valleys move the limits by 1 to 3 km (NASA's page gives this range), and the centre of the Moon's figure is offset from its centre of mass by about 0.5 km. Observers who want the limits to better than a kilometre use limb-profile corrections from lunar laser altimetry. For choosing where to stand, both calculations give the same path.

## Cross-check without the fundamental plane

The Besselian mapping can be checked with a direct 3-D calculation that does not use the fundamental plane or the polynomial fit: intersect the Sun–Moon line with the WGS84 ellipsoid (the method of the [Eclipse](../Eclipse/) tutorial), using the same apparent Sun and light-time-corrected Moon.

In [ ]:
def direct_central_line(times):
    sun, moon = sun_moon_itrf(times)
    u = (moon - sun) / np.linalg.norm(moon - sun, axis=1)[:, None]  # direction of the light
    a, b = moon * STRETCH, u * STRETCH
    qa, qb, qc = np.sum(b * b, 1), np.sum(a * b, 1), np.sum(a * a, 1) - 1
    s = (-qb - np.sqrt(qb**2 - qa * qc)) / qa  # first intersection along the light
    return (moon + s[:, None] * u) * AE


times = [sk.time(2024, 4, 8, 17, 0, 0) + sk.duration.from_minutes(m) for m in range(0, 150, 5)]
direct = direct_central_line(times)
besselian = central_line(bes, np.array([bes.hours(t) for t in times]))
print(f"max difference, direct vs Besselian central line: {np.max(np.linalg.norm(direct - besselian, axis=1)):.1f} m")

The two agree to metres: the polynomial fit and the mapping lose nothing at the scale of the map.

## Another date: 12 August 2045

The next total eclipses visible from the contiguous United States are on 23 August 2044, at sunset in Montana and North Dakota after crossing Canada, and on 12 August 2045, which crosses the country from northern California to Florida with up to 6 minutes of totality.

The code above only needs a reference epoch $t_0$ near greatest eclipse. `eclipse_t0` finds one from the elements themselves: it scans the day for the minimum distance of the shadow axis from the Earth's centre, $\sqrt{x^2+y^2}$ (the eclipse's $\gamma$), and rounds to the nearest hour of TT.

Earth orientation for 2045 is not known yet. `satkit` holds the last UT1−UTC value of its table (and warns; the warning is turned off below), and future leap seconds are unknown too, so $\Delta T$ in 2045 is uncertain by several seconds. That moves the predicted path east or west by a few kilometres; its shape and width are not affected.

In [ ]:
def eclipse_t0(year, month, day):
    # Whole hour of TT nearest to the minimum of sqrt(x^2 + y^2) on the given date
    start = sk.time(year, month, day)
    times = [start + sk.duration.from_minutes(m) for m in range(0, 1440, 10)]
    e = raw_elements(times)
    t_min = times[int(np.argmin(np.hypot(e["x"], e["y"])))]
    jd = t_min.to_jd(sk.timescale.TT)
    return sk.time.from_jd(np.round((jd - 0.5) * 24) / 24 + 0.5, sk.timescale.TT)


sk.frametransform.disable_eop_time_warning()
bes2045 = Besselian.fit(eclipse_t0(2045, 8, 12))
gamma, t_ge = greatest_eclipse(bes2045)
print(f"t0 = {bes2045.t0} UTC; greatest eclipse at {t_ge.strftime('%H:%M:%S')} UTC, gamma = {gamma:.4f}")
path2045 = compute_path(bes2045, grid=(np.arange(-136, -62, 0.1), np.arange(17, 52, 0.1)))
print(f"longest totality on the map: {fmt_duration(np.nanmax(path2045['grid'][2]))}")

CITIES_2045 = [
    ("Reno", 39.5296, -119.8138, (0.4, 0.8, "left")),
    ("Salt Lake City", 40.7608, -111.8910, (0.4, 0.8, "left")),
    ("Denver", 39.7392, -104.9903, (0.5, 0.6, "left")),
    ("Tulsa", 36.1540, -95.9928, (0.5, 0.6, "left")),
    ("Little Rock", 34.7465, -92.2896, (0.5, 0.6, "left")),
    ("Atlanta", 33.7490, -84.3880, (0.5, 0.3, "left")),
    ("Orlando", 28.5384, -81.3789, (0.5, 0.5, "left")),
    ("Miami", 25.7617, -80.1918, (0.5, 0.2, "left")),
]

fig, ax = plt.subplots(figsize=(10, 6.8), subplot_kw={"projection": LAMBERT})
base_map(ax, [-125, -70, 21, 49])
draw_path(ax, path2045, CITIES_2045, outline_minutes=0)
ax.legend(loc="lower left", fontsize=8.5, framealpha=0.9)
ax.set_title("Total solar eclipse of 12 August 2045: path of totality")
plt.show()